# RSNA Knee Abnormality Detection - EDA

This notebook builds a practical exploratory data analysis workflow for the Kaggle competition dataset.

It covers:
- study-level label and report analysis (`train.csv`)
- series-level protocol analysis (`train_series.csv`)
- sampled DICOM metadata and pixel checks (`train_series/`)
- quick checks to validate modeling readiness

In [ ]:
from __future__ import annotations

from pathlib import Path
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 120)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

In [ ]:
def resolve_data_root() -> Path:
    """Finds dataset root in Kaggle or local workspace."""
    candidates = [
        Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
        Path.cwd(),
        Path.cwd() / "data",
    ]
    for candidate in candidates:
        if (candidate / "train.csv").exists() or (candidate / "sample_submission.csv").exists():
            return candidate
    return Path.cwd()

data_root = resolve_data_root()
print(f"Data root: {data_root.resolve()}")
for name in ["train.csv", "train_series.csv", "test.csv", "test_series.csv", "sample_submission.csv", "train_series"]:
    print(f"{name:20s} -> {(data_root / name).exists()}")

In [ ]:
def read_csv_if_exists(path: Path) -> pd.DataFrame:
    if path.exists():
        return pd.read_csv(path)
    print(f"[WARN] Missing: {path}")
    return pd.DataFrame()


train_df = read_csv_if_exists(data_root / "train.csv")
train_series_df = read_csv_if_exists(data_root / "train_series.csv")
test_df = read_csv_if_exists(data_root / "test.csv")
test_series_df = read_csv_if_exists(data_root / "test_series.csv")
submission_df = read_csv_if_exists(data_root / "sample_submission.csv")

print("\nShapes")
for name, df in [
    ("train", train_df),
    ("train_series", train_series_df),
    ("test", test_df),
    ("test_series", test_series_df),
    ("sample_submission", submission_df),
]:
    print(f"{name:20s} {df.shape}")

## 1) Study-Level Overview

In [ ]:
train_df.

In [ ]:
if not train_df.empty:
    display(train_df.head(3))

    print("Unique studies:", train_df["StudyInstanceUID"].nunique() if "StudyInstanceUID" in train_df else "N/A")

    missing_pct = train_df.isna().mean().sort_values(ascending=False) * 100
    display(missing_pct[missing_pct > 0].to_frame("missing_%").head(20))
else:
    print("train.csv not available yet. Download the competition data to run full EDA.")

In [ ]:
EXPECTED_LABELS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture",
]

label_cols = [c for c in EXPECTED_LABELS if c in train_df.columns]
print("Detected label columns:", label_cols)

if label_cols:
    prevalence = train_df[label_cols].mean().sort_values(ascending=False)

    plt.figure(figsize=(10, 5))
    sns.barplot(x=prevalence.index, y=prevalence.values, palette="viridis")
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Positive rate")
    plt.title("Label prevalence in train.csv")
    plt.tight_layout()
    plt.show()

    display((prevalence * 100).round(2).to_frame("positive_%"))

In [ ]:
if label_cols:
    corr = train_df[label_cols].corr()

    plt.figure(figsize=(9, 7))
    sns.heatmap(corr, cmap="coolwarm", center=0, vmin=-1, vmax=1, square=True)
    plt.title("Label correlation matrix")
    plt.tight_layout()
    plt.show()

    positive_labels_per_study = train_df[label_cols].sum(axis=1)
    plt.figure(figsize=(8, 4))
    sns.countplot(x=positive_labels_per_study, color="steelblue")
    plt.title("Number of positive findings per study")
    plt.xlabel("Positive label count")
    plt.ylabel("Studies")
    plt.tight_layout()
    plt.show()

## 2) Series-Level Protocol Analysis

In [ ]:
if not train_series_df.empty:
    display(train_series_df.head(3))

    if "StudyInstanceUID" in train_series_df:
        series_per_study = train_series_df.groupby("StudyInstanceUID").size()
        plt.figure(figsize=(8, 4))
        sns.histplot(series_per_study, bins=30, kde=True)
        plt.title("Series per study (train)")
        plt.xlabel("Number of series")
        plt.tight_layout()
        plt.show()

        display(series_per_study.describe().to_frame("series_per_study"))

    for col in ["Anatomical_Plane", "Fluid_Sensitive", "Fat_Suppression"]:
        if col in train_series_df:
            plt.figure(figsize=(7, 4))
            vc = train_series_df[col].astype(str).value_counts()
            sns.barplot(x=vc.index, y=vc.values, palette="mako")
            plt.title(f"{col} distribution")
            plt.ylabel("Count")
            plt.tight_layout()
            plt.show()

In [ ]:
if (not train_df.empty) and (not train_series_df.empty) and label_cols:
    merged = train_series_df.merge(
        train_df[["StudyInstanceUID", *label_cols]],
        on="StudyInstanceUID",
        how="left",
    )

    if "Anatomical_Plane" in merged:
        plane_label_pos = merged.groupby("Anatomical_Plane")[label_cols].mean().T
        plt.figure(figsize=(10, 5))
        sns.heatmap(plane_label_pos, cmap="YlGnBu", annot=False)
        plt.title("Positive label rates by anatomical plane")
        plt.tight_layout()
        plt.show()

## 3) DICOM-Level Checks (Sampled)

This section samples a limited number of DICOM slices so EDA stays fast.

In [ ]:
try:
    import pydicom
except ImportError:
    raise ImportError("Please install pydicom: pip install pydicom")

train_series_dir = data_root / "train_series"
if train_series_dir.exists():
    dcm_paths = list(train_series_dir.rglob("*.dcm"))
    print(f"Total DICOM files discovered: {len(dcm_paths):,}")
else:
    dcm_paths = []
    print("train_series directory not found.")

SAMPLE_DCM = 400
sampled_dcm_paths = random.sample(dcm_paths, min(SAMPLE_DCM, len(dcm_paths))) if dcm_paths else []
print(f"Sampled {len(sampled_dcm_paths)} DICOM files")

In [ ]:
meta_rows = []
for path in sampled_dcm_paths:
    try:
        ds = pydicom.dcmread(path, stop_before_pixels=True, force=True)
        meta_rows.append({
            "path": str(path),
            "Rows": getattr(ds, "Rows", np.nan),
            "Columns": getattr(ds, "Columns", np.nan),
            "BitsStored": getattr(ds, "BitsStored", np.nan),
            "PixelRepresentation": getattr(ds, "PixelRepresentation", np.nan),
            "PhotometricInterpretation": getattr(ds, "PhotometricInterpretation", None),
            "TransferSyntaxUID": str(getattr(getattr(ds, "file_meta", None), "TransferSyntaxUID", "Unknown")),
            "SliceThickness": getattr(ds, "SliceThickness", np.nan),
        })
    except Exception:
        continue

meta_df = pd.DataFrame(meta_rows)
print("Metadata rows:", len(meta_df))
display(meta_df.head(3))

In [ ]:
if not meta_df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    sns.histplot(meta_df["Rows"].dropna(), bins=30, ax=axes[0], color="teal")
    axes[0].set_title("Rows")

    sns.histplot(meta_df["Columns"].dropna(), bins=30, ax=axes[1], color="slateblue")
    axes[1].set_title("Columns")

    sns.histplot(meta_df["SliceThickness"].dropna(), bins=30, ax=axes[2], color="tomato")
    axes[2].set_title("Slice Thickness")

    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 4))
    ts_counts = meta_df["TransferSyntaxUID"].value_counts().head(10)
    sns.barplot(x=ts_counts.index, y=ts_counts.values, palette="crest")
    plt.xticks(rotation=45, ha="right")
    plt.title("Most common Transfer Syntax UIDs (sample)")
    plt.tight_layout()
    plt.show()

In [ ]:
def load_normalized_pixel_array(path: Path) -> np.ndarray:
    ds = pydicom.dcmread(path, force=True)
    arr = ds.pixel_array.astype(np.float32)
    arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-6)
    return arr


if sampled_dcm_paths:
    preview_paths = random.sample(sampled_dcm_paths, min(6, len(sampled_dcm_paths)))
    fig, axes = plt.subplots(2, 3, figsize=(12, 8))

    for ax, p in zip(axes.flat, preview_paths):
        try:
            img = load_normalized_pixel_array(p)
            ax.imshow(img, cmap="gray")
            ax.set_title(p.parent.name[:10])
            ax.axis("off")
        except Exception:
            ax.axis("off")
            ax.set_title("Read error")

    plt.suptitle("Random DICOM slice previews")
    plt.tight_layout()
    plt.show()
else:
    print("No DICOM files available for preview.")

## 4) Report Text Quick Profiling

In [ ]:
if "Report" in train_df:
    report_series = train_df["Report"].fillna("").astype(str)
    report_len = report_series.str.len()

    plt.figure(figsize=(9, 4))
    sns.histplot(report_len, bins=50, kde=True, color="purple")
    plt.title("Report character-length distribution")
    plt.xlabel("Characters")
    plt.tight_layout()
    plt.show()

    # Simple heuristic: fraction of ascii characters as a rough language diversity signal.
    def ascii_fraction(text: str) -> float:
        if not text:
            return np.nan
        ascii_chars = sum(1 for ch in text if ord(ch) < 128)
        return ascii_chars / len(text)

    ascii_frac = report_series.sample(min(5000, len(report_series)), random_state=RANDOM_SEED).map(ascii_fraction)
    plt.figure(figsize=(8, 4))
    sns.histplot(ascii_frac.dropna(), bins=40, color="darkorange")
    plt.title("ASCII character fraction in reports (sample)")
    plt.xlabel("ASCII fraction")
    plt.tight_layout()
    plt.show()

    display(report_series.sample(min(5, len(report_series)), random_state=RANDOM_SEED))

## 5) Next Modeling-Oriented Checks

Recommended follow-up analyses:
- split by study/site proxies and verify label prevalence drift
- inspect orientation and spacing consistency per `Anatomical_Plane`
- build per-series sampling strategy for training (plane + fluid/fat balance)
- evaluate report-derived weak labels for unlabeled studies